# Batch 1 — control, E12, E10, E11, E16, E18 (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all; about 80–90 minutes. Keep the tab open.
If Colab disconnects, Run all again: finished runs are skipped.

Every run is the E03 configuration with **one** change (fixed before any run, experiment ledger):

| Run | Change vs E03 |
|---|---|
| control | none — E03 again in this session; also gives **E12** (best epoch kept per horizon, no change to training) |
| E10 | per-window normalisation (RevIN) off |
| E11 | classification only (no return-forecast loss) |
| E16 | label smoothing 0.1 |
| E18 | no sector label |

Seeds 20261015/16/17. Outputs in `MyDrive/Senior/results/batch1/`, summary `batch1_summary.json`. Test block stays sealed.


In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os
src = open('scripts/baselines_deep.py', encoding='utf-8').read()
assert 'OPT = dict(revin=True' in src and 'val_per_horizon_best' in src, 'stop: push the latest baselines_deep.py from your PC'
assert os.path.exists('scripts/compare_batch.py'), 'stop: push scripts/compare_batch.py from your PC'
print('baselines_deep.py', hashlib.sha256(open('scripts/baselines_deep.py', 'rb').read()).hexdigest()[:12])
B1 = f'{OUT_DIR}/batch1'
os.makedirs(B1, exist_ok=True)


In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Pipeline and look-ahead tests; the notebook stops here if either fails
import subprocess
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed'


In [ ]:
# Switch check (stops on mismatch): weight counts, RevIN removal, classification-only loss, label smoothing
import sys, torch
sys.path.insert(0, 'scripts'); sys.path.insert(0, 'src')
import baselines_deep as BD
def n(m): return sum(p.numel() for p in m.parameters())
BD.OPT.update(revin=True, cls_only=False, label_smoothing=0.0)
base = BD.make_model('multiscale_sector', 32, 18, 0.2)
BD.OPT.update(revin=False)
norev = BD.make_model('multiscale_sector', 32, 18, 0.2)
print('weights: with RevIN', n(base), '| without', n(norev), '| no sector', n(BD.make_model('multiscale', 32, 0, 0.2)))
assert n(base) == 472_636 and n(norev) == 472_636 - 64 and norev.revin is None, 'stop: RevIN switch wrong'
torch.manual_seed(0)
out = base(torch.randn(8, 60, 32), torch.randint(0, 19, (8,)))
y, r = torch.randint(0, 3, (8, 3)), torch.randn(8, 3)
BD.OPT.update(revin=True, cls_only=True, label_smoothing=0.0)
l_cls, _ = BD.make_loss()(out, y, r)
expect = torch.nn.functional.cross_entropy(out['trend_logits'].reshape(-1, 3), y.reshape(-1))
assert torch.allclose(l_cls, expect), 'stop: classification-only loss is not plain cross-entropy'
BD.OPT.update(cls_only=False, label_smoothing=0.1)
assert abs(BD.make_loss().ce.label_smoothing - 0.1) < 1e-12, 'stop: label smoothing not applied'
BD.OPT.update(revin=True, cls_only=False, label_smoothing=0.0)
print('switch check passed')


In [ ]:
# Smoke run (2 epochs, 1 seed, scratch file): the per-horizon (E12) output must be written
import json, subprocess
subprocess.run(['python', 'scripts/baselines_deep.py', '--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv',
                '--extra-features', 'e03', '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '2', '--patience', '1',
                '--seeds', '20261015', '--out', '/content/smoke.json'], check=True, capture_output=True)
run = json.load(open('/content/smoke.json'))['models']['multiscale_sector']['runs'][0]
assert 'val_per_horizon_best' in run and len(run['best_epoch_per_horizon']) == 3, 'stop: per-horizon output missing'
print('smoke run passed: best epoch per horizon', run['best_epoch_per_horizon'])


In [ ]:
# The batch: 5 configurations x 3 seeds. Finished runs are skipped on a rerun.
import json, subprocess, sys
COMMON = ['--news', 'data/derived/sentiment_scores.csv', '--extra-features', 'e03',
          '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '30', '--patience', '10']
RUNS = [('control', 'multiscale_sector', []),
        ('e10_no_revin', 'multiscale_sector', ['--no-revin']),
        ('e11_cls_only', 'multiscale_sector', ['--cls-only']),
        ('e16_label_smoothing', 'multiscale_sector', ['--label-smoothing', '0.1']),
        ('e18_no_sector', 'multiscale', [])]
def done(path, model):
    try:
        return len(json.load(open(path))['models'][model]['runs']) == 3
    except Exception:
        return False
for label, model, extra in RUNS:
    out = f'{B1}/{label}.json'
    if done(out, model):
        print(f'=== {label}: already done'); continue
    print(f'=== {label}'); sys.stdout.flush()
    subprocess.run(['python', 'scripts/baselines_deep.py', '--models', model, *COMMON, *extra, '--out', out], check=True)


In [ ]:
# Summary: each change vs the same-session control, E12 (per-horizon best epoch), gap to the pass bar,
# and the control's reproducibility against the earlier E03 run (same seeds)
!python scripts/compare_batch.py --control "$OUT_DIR/batch1/control.json" --reference "$OUT_DIR/main_model_e03.json" \
    --run E10 "$OUT_DIR/batch1/e10_no_revin.json" --run E11 "$OUT_DIR/batch1/e11_cls_only.json" \
    --run E16 "$OUT_DIR/batch1/e16_label_smoothing.json" --run E18 "$OUT_DIR/batch1/e18_no_sector.json" \
    --out "$OUT_DIR/batch1/batch1_summary.json"
